# fact_ventas
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Gold (nivel 2) · hecho Ventas |
| Grano | un ítem de pedido (permite analizar por producto y categoría sin duplicar). |
| Fuentes | silver.items_pedido + silver.pedidos · dim_clientes · dim_productos |
| Decisiones | - El cliente se une a la versión SCD2 vigente en la fecha del pedido. - Ventas = suma de ítems (cantidad × precio − descuento), no el total de cabecera, que en algunos pedidos legacy no cuadra. - es_venta_efectiva: pedidos pagados, enviados o entregados. - Liquid clustering por fecha y cliente (las consultas más frecuentes). |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun_gold import ESTADOS_VENTA_EFECTIVA, cliente_vigente_en, leer_silver, sk, sk_fecha

## 2. Tabla `fact_ventas`
Ventas al grano de ítem de pedido, con el segmento del cliente vigente al comprar.

In [ ]:
@dlt.table(name="fact_ventas", cluster_by=["sk_fecha", "sk_cliente"],
           comment="Ventas al grano de ítem de pedido, con el segmento del cliente vigente al comprar.")
def fact_ventas():
    items = leer_silver("items_pedido")
    pedidos = leer_silver("pedidos").select("id_pedido", "id_cliente", "fecha_pedido", "canal",
                                            F.col("estado").alias("estado_pedido"))
    productos = dlt.read("dim_productos").select("sk_producto", "id_producto")
    ventas = items.join(pedidos, "id_pedido", "inner").join(productos, "id_producto", "left")
    ventas = cliente_vigente_en(ventas, dlt.read("dim_clientes"), "id_cliente", "fecha_pedido")
    return ventas.select(
        sk_fecha("fecha_pedido").alias("sk_fecha"), "sk_cliente", "sk_producto", sk("canal").alias("sk_canal"),
        "id_pedido", "id_item", "id_cliente", "id_producto", "fecha_pedido", "estado_pedido",
        "cantidad", "precio_unitario", "descuento", "monto_item",
        F.col("estado_pedido").isin(ESTADOS_VENTA_EFECTIVA).alias("es_venta_efectiva"))